# CER — Cluster Experiment Runner Demo

This notebook walks through the full CER workflow:
1. Commit & push your experiment code
2. Submit to a SLURM cluster
3. Monitor job status
4. Query results from W&B
5. Visualize metrics

## 0. Setup

Make sure CER is installed and configured:
```bash
pip install cluster-experiment-runner
```

Configure `~/.config/cer/cer.yaml` with your cluster and W&B settings.

In [ ]:
import subprocess, json, os

# If cer is not on PATH, set the full path here:
CER = "cer"

!{CER} --help

## 1. Commit & Push

CER uses **commit = experiment**. Every experiment is tied to a git commit for full reproducibility.

In [ ]:
# Check repo status
!git status
print("---")
!git log --oneline -5

In [ ]:
# Uncomment to commit and push changes:
# !git add -A && git commit -m "experiment: describe what you changed" && git push

In [ ]:
# Get the current commit hash
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"Commit: {COMMIT}")
print(f"Short:  {COMMIT[:8]}")

## 2. Submit Experiment

CER will SSH to the cluster, create a worktree at this commit, and submit a SLURM job.

If `experiment.def` changed since the last run, the Singularity container is automatically rebuilt.

In [ ]:
!{CER} submit {COMMIT}

In [ ]:
# Paste the job ID from above
JOB_ID = input("Enter job ID: ")

## 3. Check Status

Queries SLURM via SSH. Run multiple times to watch transitions: `PENDING` → `RUNNING` → `COMPLETED`.

In [ ]:
!{CER} status {JOB_ID}

In [ ]:
# JSON output (for programmatic use)
!{CER} status {JOB_ID} --json

## 4. List All Experiments

Shows all tracked experiments. Active jobs get their SLURM status refreshed automatically.

In [ ]:
!{CER} list

## 5. Query W&B Results

CER finds the W&B run by matching the commit hash tag, then pulls metrics via the W&B API.

In [ ]:
# Summary metrics
!{CER} results {JOB_ID}

In [ ]:
# Full training history
!{CER} results {JOB_ID} --history

In [ ]:
# Filter specific metrics
!{CER} results {JOB_ID} --history --key train/loss --key val/loss

## 6. Programmatic Access (JSON)

Use `--json` output for automated pipelines or agent consumption.

In [ ]:
raw = subprocess.check_output([CER, "results", JOB_ID, "--json", "--history"]).decode()
# Skip any wandb log lines before the JSON
json_str = raw[raw.index("{"):]
data = json.loads(json_str)

print(f"Run:     {data['wandb']['run_name']}")
print(f"State:   {data['wandb']['state']}")
print(f"URL:     {data['wandb']['url']}")
print(f"Summary: {data['wandb']['summary']}")
print(f"History: {len(data.get('history', []))} steps")

## 7. Plot Results

In [ ]:
import matplotlib.pyplot as plt

history = data["history"]

# Adapt these keys to your experiment's logged metrics
steps = range(len(history))
train_loss = [h.get("train/loss") for h in history]
val_loss = [h.get("val/loss") for h in history]

plt.figure(figsize=(10, 5))
if any(v is not None for v in train_loss):
    plt.plot(steps, train_loss, label="train/loss", marker="o")
if any(v is not None for v in val_loss):
    plt.plot(steps, val_loss, label="val/loss", marker="s")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title(f"Experiment {JOB_ID} (commit {COMMIT[:8]})")
plt.legend()
plt.grid(True)
plt.show()

## 8. Cancel an Experiment

Runs `scancel` on the cluster.

In [ ]:
# Uncomment to cancel a running job:
# !{CER} cancel {JOB_ID}

## 9. MCP Server (for AI Agents)

CER can run as an MCP server, exposing all commands as tools for AI agents.

```bash
# Start the MCP server on the host (has SSH access)
cer-mcp

# The agent connects to http://localhost:8000/sse
# Available tools: submit, status, cancel, results, list_experiments
```

The agent runs inside a Singularity container (sandboxed, no SSH) and calls CER tools over the network.

---

## Command Reference

| Command | Description |
|---------|-------------|
| `cer submit <commit>` | Submit experiment to SLURM cluster |
| `cer status <job_id>` | Check SLURM job status |
| `cer status <job_id> --json` | Status as JSON |
| `cer results <job_id>` | W&B summary metrics |
| `cer results <job_id> --history` | Full metric history |
| `cer results <job_id> --key <metric>` | Filter specific metrics |
| `cer results <job_id> --json` | Results as JSON |
| `cer cancel <job_id>` | Cancel running job |
| `cer list` | List all experiments |
| `cer local` | Run experiment locally in container |
| `cer local --build` | Rebuild container, then run |
| `cer local --shell` | Open shell inside container |
| `cer-mcp` | Start MCP server for AI agents |